# Week 4: Multi-Tool Assistant, Diabetes Glucose Assistant (Synthetic Data)

A tool-calling assistant for reviewing synthetic patient glucose data. All patient data (P001, P002, P003) is fabricated for this assignment and there are no real patients or medical advice.

**Three tools:**
- `get_daily_summary`: retrieves daily mean glucose for a patient over a fixed
  period (7, 14, or 30 days), in mg/dL or mmol/L.
- `classify_glucose`: categorizes a single glucose value (very_low, low,
  in_range, high, very_high), using the international consensus thresholds
  (Battelino et al. 2019, Diabetes Care 42(8):1593-1603).
- `run_python`: guarded arithmetic evaluator, used by the model to average or
  convert values returned by the other tools. Allowlist-checked before
  execution, with a thread-based timeout. Blocks filesystem access, network
  calls, process execution, imports, and any non-arithmetic input.

**What's demonstrated below:**
- Part 1: live request/response loop against the Gemini OpenAI-compatible endpoint
- Part 2: the guarded `run_python` and what it permits and blocks
- Part 3: a two-tool query and a three-tool query, both with full call logs
- Part 4: a schema failure (`classify_glucose` called with  `unit: 'mg'` instead of `'mg/dL'`) and how the model recovered from the structured error

Live model calls run only when `GEMINI_API_KEY` is set in a local `.env` file.
It is not committed. Scripted checks below run with no API key and no cost.

In [1]:
import os, ast, operator, json, math
from dotenv import load_dotenv
from openai import OpenAI
import concurrent.futures
import re

load_dotenv()

GEMINI_KEY = os.environ.get('GEMINI_API_KEY', '').strip()
HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)


MODEL = 'gemini-3.5-flash'   # swap for the Flash name in your Week 0 guide if this errors

client = OpenAI(
    api_key=GEMINI_KEY,
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
)
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with the word ok.'}],
)
print(r.choices[0].message.content)

GEMINI_API_KEY set: True
ok


In [2]:
# Local function definitions with JSON schemas.
TOOLS = [
  {
    'name': 'get_daily_summary',
    'description': (
      'Get the daily mean glucose values for one synthetic patient over a recent period. '
      'Returns a list of daily means in the requested unit, oldest day first. '
      'Use this first whenever a question needs a patient\'s glucose numbers.'
    ),
    'parameters': {
      'type': 'object',
      'properties': {
        'patient_id':  {'type': 'string',  'enum': ['P001', 'P002', 'P003']},
        'period_days': {'type': 'integer', 'enum': [7, 14, 30]},
        'unit':        {'type': 'string',  'enum': ['mg/dL', 'mmol/L']},
      },
      'required': ['patient_id', 'period_days', 'unit'],
      'additionalProperties': False,
    },
  },
  {
    'name': 'classify_glucose',
    'description': (
      'Classify a single glucose value into a range category: very_low, low, in_range, '
      'high or very_high. Use for questions about whether a specific value is in range. '
      'Does not do arithmetic and gives no treatment advice.'
    ),
    'parameters': {
      'type': 'object',
      'properties': {
        'value': {'type': 'number'},
        'unit':  {'type': 'string', 'enum': ['mg/dL', 'mmol/L']},
      },
      'required': ['value', 'unit'],
      'additionalProperties': False,
    },
  },
  {
    'name': 'run_python',
    'description': (
      'Guarded code-runner. Evaluates ONE arithmetic expression made only of numbers and '
      '+ - * / ** and parentheses, for example averaging values or converting units. '
      'Write the actual numbers into the expression; variables and function calls are '
      'not allowed. Returns the numeric result.'
    ),
    'parameters': {
      'type': 'object',
      'properties': {
        'expression': {'type': 'string'},
      },
      'required': ['expression'],
      'additionalProperties': False,
    },
  },
]
print('tools:', [t['name'] for t in TOOLS])

tools: ['get_daily_summary', 'classify_glucose', 'run_python']


In [3]:
# Synthetic patient data and the two non-runner tools.
# All data below is invented for this assignment. Not real patients & not medical advice.

# 30 daily mean glucose values in mg/dL per patient, oldest first, so any
# period_days in {7, 14, 30} can slice from the end.
PATIENT_DATA = {
    'P001': [142, 138, 155, 121, 160, 133, 145, 150, 128, 140,
             162, 135, 148, 152, 130, 144, 158, 126, 139, 151,
             147, 133, 156, 142, 149, 137, 153, 141, 146, 150],
    'P002': [98, 105, 88, 250, 210, 95, 102, 91, 99, 108,
             94, 220, 205, 97, 103, 89, 101, 96, 110, 92,
             98, 195, 188, 100, 93, 107, 91, 99, 104, 96],
    'P003': [180, 175, 190, 168, 172, 185, 178, 182, 176, 190,
             165, 179, 183, 177, 188, 170, 181, 174, 186, 173,
             179, 184, 171, 187, 175, 180, 178, 182, 176, 185],
}

MGDL_PER_MMOL = 18.0182  # 1 mmol/L = 18.0182 mg/dL

def _convert(value_mgdl, unit):
    if unit == 'mg/dL':
        return round(value_mgdl, 1)
    if unit == 'mmol/L':
        return round(value_mgdl / MGDL_PER_MMOL, 2)
    raise ValueError(f'unsupported unit: {unit}')

def get_daily_summary(patient_id, period_days, unit):
    if patient_id not in PATIENT_DATA:
        raise ValueError(f'unknown patient_id: {patient_id}')
    days_mgdl = PATIENT_DATA[patient_id][-period_days:]
    return {
        'patient_id': patient_id,
        'period_days': period_days,
        'unit': unit,
        'daily_means': [_convert(v, unit) for v in days_mgdl],
    }

# Thresholds in mg/dL, from the international consensus on time-in-range
# categories (Battelino et al. 2019, Diabetes Care)

_THRESHOLDS_MGDL = {
    'very_low':  (float('-inf'), 54),
    'low':       (54, 70),
    'in_range':  (70, 180),
    'high':      (180, 250),
    'very_high': (250, float('inf')),
}

def classify_glucose(value, unit):
    value_mgdl = value if unit == 'mg/dL' else value * MGDL_PER_MMOL
    for label, (lo, hi) in _THRESHOLDS_MGDL.items():
        if lo <= value_mgdl < hi:
            return {'value': value, 'unit': unit, 'category': label}
    raise ValueError(f'could not classify value: {value}')

IMPL = {
    'get_daily_summary': get_daily_summary,
    'classify_glucose': classify_glucose,
}

# Quick scripted checks, no API calls
print(get_daily_summary('P002', 7, 'mg/dL'))
print(get_daily_summary('P001', 14, 'mmol/L'))
print(classify_glucose(45, 'mg/dL'))
print(classify_glucose(6.5, 'mmol/L'))
print(classify_glucose(300, 'mg/dL'))

{'patient_id': 'P002', 'period_days': 7, 'unit': 'mg/dL', 'daily_means': [100, 93, 107, 91, 99, 104, 96]}
{'patient_id': 'P001', 'period_days': 14, 'unit': 'mmol/L', 'daily_means': [8.77, 6.99, 7.71, 8.38, 8.16, 7.38, 8.66, 7.88, 8.27, 7.6, 8.49, 7.83, 8.1, 8.32]}
{'value': 45, 'unit': 'mg/dL', 'category': 'very_low'}
{'value': 6.5, 'unit': 'mmol/L', 'category': 'in_range'}
{'value': 300, 'unit': 'mg/dL', 'category': 'very_high'}


In [4]:
# Arithmetic-only interpreter.
_OPS = {ast.Add:operator.add, ast.Sub:operator.sub, ast.Mult:operator.mul, ast.Div:operator.truediv, ast.Pow:operator.pow, ast.USub:operator.neg}
def _safe(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int,float): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe(node.left), _safe(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe(node.operand))
    raise ValueError('only arithmetic is allowed')   # blocks names, calls, imports, attributes

# Part 1: allowlist check, before anything runs.
_ALLOWED_CHARS = re.compile(r'^[0-9\.\s\+\-\*\/\(\)]+$')
_MAX_EXPR_LEN = 200
def check_allowed(expression):
    if len(expression) > _MAX_EXPR_LEN:
        raise ValueError(f'expression too long (max {_MAX_EXPR_LEN} chars)')
    if not _ALLOWED_CHARS.match(expression):
        raise ValueError('expression contains characters outside allowed arithmetic set')

def run_python(expression, timeout_seconds=2):
    check_allowed(expression)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(_safe, ast.parse(expression, mode='eval').body)
        try:
            return future.result(timeout=timeout_seconds)
        except concurrent.futures.TimeoutError:
            raise TimeoutError(f'expression took longer than {timeout_seconds}s')

# Merge into the IMPL dict from step 2 (do NOT overwrite it with a fresh dict).
IMPL['run_python'] = run_python

class ToolArgError(Exception): pass
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k,v in args.items():
        p = spec['properties'].get(k)
        if p is None: raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str): raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float): raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v): raise ToolArgError(f'{k} must be finite')
        elif p['type'] == 'integer':
            if type(v) is not int: raise ToolArgError(f'{k} must be an integer')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']: raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {'ok':True,'tool':name,'output':output}
    except ToolArgError as e:
        return {'ok':False,'tool':name,'error_type':'invalid_arguments','message':str(e)}
    except Exception as e:
        return {'ok':False,'tool':name,'error_type':'execution_error','message':str(e)}

print('happy path:', dispatch('run_python', {'expression':'(142+138+155)/3'}))
print('guarded (should fail):', dispatch('run_python', {'expression':'__import__("os").system("echo hi")'}))

happy path: {'ok': True, 'tool': 'run_python', 'output': 145.0}
guarded (should fail): {'ok': False, 'tool': 'run_python', 'error_type': 'execution_error', 'message': 'expression contains characters outside allowed arithmetic set'}


## Parts 1, 3, and 4: dispatch, evaluation, and recovery

In [5]:
scripted = [
  ('classify_glucose', {'value': 45, 'unit': 'mg/dL'}),
  ('get_daily_summary', {'patient_id': 'P002', 'period_days': 7, 'unit': 'mg/dL'}),
  ('run_python', {'expression': '6*7'}),
]
for name, args in scripted:
    r = dispatch(name, args)
    tag = 'OK ' if r['ok'] else 'ERR'
    print(f'[{tag}] {name}({args}) -> {json.dumps(r, allow_nan=False)}')

[OK ] classify_glucose({'value': 45, 'unit': 'mg/dL'}) -> {"ok": true, "tool": "classify_glucose", "output": {"value": 45, "unit": "mg/dL", "category": "very_low"}}
[OK ] get_daily_summary({'patient_id': 'P002', 'period_days': 7, 'unit': 'mg/dL'}) -> {"ok": true, "tool": "get_daily_summary", "output": {"patient_id": "P002", "period_days": 7, "unit": "mg/dL", "daily_means": [100, 93, 107, 91, 99, 104, 96]}}
[OK ] run_python({'expression': '6*7'}) -> {"ok": true, "tool": "run_python", "output": 42}


In [6]:
def run_agent(user_query, max_rounds=6):
    messages = [{'role': 'user', 'content': user_query}]
    call_log = []

    for round_num in range(max_rounds):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages,
            tools=[{'type': 'function', 'function': t} for t in TOOLS],
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content, call_log

        messages.append(msg)
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
                result = dispatch(tc.function.name, args)
            except json.JSONDecodeError as e:
                result = {'ok': False, 'tool': tc.function.name,
                          'error_type': 'invalid_arguments', 'message': str(e)}
            call_log.append({'round': round_num, 'tool': tc.function.name,
                              'args': tc.function.arguments, 'ok': result.get('ok')})
            messages.append({'role': 'tool', 'tool_call_id': tc.id,
                              'content': json.dumps(result, allow_nan=False)})

    return None, call_log

### Adding 6 diff queries/examples to test model

In [7]:
answer, log = run_agent("What was P001's average glucose over the last 7 days in mg/dL?")
print(answer)
print()
for entry in log:
    print(entry)

Over the last 7 days, P001's daily mean glucose values in mg/dL were: 142, 149, 137, 153, 141, 146, and 150. 

Their average glucose over this period was **145.43 mg/dL** (calculated as $(142 + 149 + 137 + 153 + 141 + 146 + 150) \div 7$).

{'round': 0, 'tool': 'get_daily_summary', 'args': '{"patient_id":"P001","unit":"mg/dL","period_days":7}', 'ok': True}
{'round': 1, 'tool': 'run_python', 'args': '{"expression":"(142 + 149 + 137 + 153 + 141 + 146 + 150) / 7"}', 'ok': True}


In [8]:
answer2, log2 = run_agent("Was P003's average glucose over the last 14 days in range?")
print(answer2)
print()
for entry in log2:
    print(entry)

Yes, P003's average glucose over the last 14 days was **in range**. 

Here are the details of the calculation:
* **Daily Mean Glucose Values (mg/dL):** 181, 174, 186, 173, 179, 184, 171, 187, 175, 180, 178, 182, 176, 185
* **Calculated Average:** ~179.36 mg/dL
* **Classification:** In Range

{'round': 0, 'tool': 'get_daily_summary', 'args': '{"period_days":14,"unit":"mg/dL","patient_id":"P003"}', 'ok': True}
{'round': 1, 'tool': 'run_python', 'args': '{"expression":"(181 + 174 + 186 + 173 + 179 + 184 + 171 + 187 + 175 + 180 + 178 + 182 + 176 + 185) / 14"}', 'ok': True}
{'round': 2, 'tool': 'classify_glucose', 'args': '{"unit":"mg/dL","value":179.36}', 'ok': True}


In [9]:
answer3, log3 = run_agent("What's P4's average glucose recently, in mmol, if the readings are in mg?")
print(answer3)
print()
for entry in log3:
    print(entry)

I only have data for patients **P001**, **P002**, and **P003**. 

If you meant one of these patients, please let me know which one, and I will gladly calculate their recent average glucose for you!



In [10]:
answer4, log4 = run_agent("What was P001's average glucose over the last 10 days?")
print(answer4)
print()
for entry in log4:
    print(entry)

To find patient P001's average glucose over the last 10 days, we first retrieve the daily means for the last 14 days (oldest first):
`[158, 126, 139, 151, 147, 133, 156, 142, 149, 137, 153, 141, 146, 150]` mg/dL

Taking the 10 most recent days (the last 10 values in the list):
* Day 5: 147 mg/dL
* Day 6: 133 mg/dL
* Day 7: 156 mg/dL
* Day 8: 142 mg/dL
* Day 9: 149 mg/dL
* Day 10: 137 mg/dL
* Day 11: 153 mg/dL
* Day 12: 141 mg/dL
* Day 13: 146 mg/dL
* Day 14: 150 mg/dL

The sum of these 10 values is **1,454 mg/dL**. 

Dividing by 10, P001's average glucose over the last 10 days is **145.4 mg/dL** (or approximately **8.1 mmol/L**).

{'round': 0, 'tool': 'get_daily_summary', 'args': '{"unit":"mg/dL","patient_id":"P001","period_days":14}', 'ok': True}
{'round': 1, 'tool': 'run_python', 'args': '{"expression":"(147 + 133 + 156 + 142 + 149 + 137 + 153 + 141 + 146 + 150) / 10"}', 'ok': True}


In [11]:
answer5, log5 = run_agent("Call the daily summary tool for patient P001 with period_days set to 10.")
print(answer5)
print()
for entry in log5:
    print(entry)

The daily summary tool only supports periods of 7, 14, or 30 days. To ensure we captured at least 10 days of data, I retrieved the **14-day summary** for patient **P001** (using mg/dL as the unit):

* **14-Day Daily Means (oldest to newest):** 158, 126, 139, 151, 147, 133, 156, 142, 149, 137, 153, 141, 146, 150 mg/dL

If you specifically need the most recent **10 days**, they are:
* **147, 133, 156, 142, 149, 137, 153, 141, 146, 150 mg/dL**

{'round': 0, 'tool': 'get_daily_summary', 'args': '{"period_days":14,"patient_id":"P001","unit":"mg/dL"}', 'ok': True}


In [12]:
answer6, log6 = run_agent("Call the classify_glucose tool with value 200 and unit set to exactly 'mg'.")
print(answer6)
print()
for entry in log6:
    print(entry)

The unit `'mg'` is not a valid option for the `classify_glucose` tool (the permitted options are `'mg/dL'` and `'mmol/L'`). 

Assuming you meant `'mg/dL'`, I have run the classification for you:
* **Value:** 200 mg/dL
* **Category:** **high**

{'round': 0, 'tool': 'classify_glucose', 'args': '{"unit":"mg","value":200}', 'ok': False}
{'round': 1, 'tool': 'classify_glucose', 'args': '{"unit":"mg/dL","value":200}', 'ok': True}
